# Uber Eats Mx Churning Prediction

In [9]:
from pathlib import Path
import pandas as pd

# V3 原始训练数据：五张表都放在仓库的 data/ 目录中
data_dir = Path('data')
print('Data directory:', data_dir.resolve())

# 读取核心表
merchant_dim_df = pd.read_csv(data_dir / 'merchant_dim.csv')
merchant_daily_df = pd.read_csv(data_dir / 'merchant_daily_sample.csv')
orders_fact_df = pd.read_csv(data_dir / 'orders_sample.csv')
support_fact_df = pd.read_csv(data_dir / 'support_fact.csv')
promotion_fact_df = pd.read_csv(data_dir / 'promotion_fact.csv')

# 先看关键表结构
for name, df in {
    'merchant_dim': merchant_dim_df,
    'merchant_daily_sample': merchant_daily_df,
    'orders_sample': orders_fact_df,
    'support_fact': support_fact_df,
    'promotion_fact': promotion_fact_df,
}.items():
    display(df.sample(3))
    print(f'\n--- {name} ---')
    print(df.shape)

Data directory: /Users/jason/Desktop/p_learn/ml/ML_Uber_Eats_Prediction_September_2026/data


,merchant_id,merchant_name,city,cuisine_type,segment,join_date,menu_ready,competitor_presence,profile_avg_rating
432,M00433,Merchant 00433,San Jose,Thai,SMB,2020-07-03,1,0,4.29
7,M00008,Merchant 00008,Fremont,Mexican,SMB,2021-03-14,1,1,4.84
839,M00840,Merchant 00840,Berkeley,Mexican,Mid-Market,2020-05-15,1,0,4.31



--- merchant_dim ---
(2000, 9)


,merchant_id,date,is_open,scheduled_hours,actual_open_hours,business_hours_consistency,temporary_closure_flag,orders_created,accepted_orders,cancelled_orders,merchant_delay_orders,avg_prep_minutes
31697,M00265,2026-02-18,1,12.27,8.99,0.733,0,0,0,0,0,NaN
112539,M00938,2026-05-11,1,10.15,7.45,0.734,0,2,2,0,0,24.17
89138,M00743,2026-05-10,1,10.69,10.69,1.000,0,0,0,0,0,NaN



--- merchant_daily_sample ---
(240000, 12)


...


--- orders_sample ---
(105095, 25)


,ticket_id,merchant_id,created_at,issue_type,resolved_flag,resolved_at,resolution_hours,merchant_satisfaction_score
5249,T0005252,M01909,2026-02-07 15:00:00,Payment,1,2026-02-07 22:08:37.043371143,7.14,4.0
4871,T0004871,M01768,2026-02-27 20:00:00,Operations,1,2026-02-28 20:36:26.582328745,24.61,4.0
551,T0000556,M00224,2026-02-22 13:00:00,Account,1,2026-02-23 09:05:26.767006396,20.09,5.0



--- support_fact ---
(5492, 8)


,promotion_id,merchant_id,promotion_type,start_date,end_date,promo_spend,promotion_return,promotion_roi
810,P0000811,M00597,Sponsored Listing,2026-04-05,2026-04-18,54.12,89.78,0.6589
938,P0000939,M00686,Free Delivery,2026-03-06,2026-03-15,162.54,338.37,1.0819
236,P0000237,M00178,Percentage Off,2026-04-11,2026-04-21,76.08,143.76,0.8896



--- promotion_fact ---
(2687, 8)


In [4]:
## Collect data

### Step 1：用 SQL 从原始表中抽取特征

先把数据整理成一个“merchant-level 的特征表”。

核心思路：
- 先从 `merchant_dim` 取商家基础信息
- 再从 `merchant_daily_sample` 计算运营活跃度
- 再从 `orders_sample` 计算 GMV、订单数、新客比、退款率等
- 再从 `support_fact` 和 `promotion_fact` 计算支持和营销信号
- 最后，将这些结果按 `merchant_id` 汇总到一张特征表

这一步就是建模前最重要的 SQL 聚合步骤。

## 订单表现

In [8]:
# 安装 pandasql（如果环境中还没有）
%pip install pandasql -q


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pandasql import sqldf
pd.set_option('display.max_columns', None)

# 先把表注册成 SQL 表
# 注意：这里我们并不是直接从 feature_mart 读，而是从原始事实表逐步抽取特征
merchant_dim_df = pd.read_csv(data_dir / 'merchant_dim.csv')
merchant_daily_df = pd.read_csv(data_dir / 'merchant_daily_sample.csv')
orders_fact_df = pd.read_csv(data_dir / 'orders_sample.csv')
support_fact_df = pd.read_csv(data_dir / 'support_fact.csv')
promotion_fact_df = pd.read_csv(data_dir / 'promotion_fact.csv')

# 示例：只抽取最近 28 天的订单表现指标
observation_date = '2026-06-30'

sql_order_features = '''
WITH params AS (
    SELECT DATE('2026-06-30') AS obs_date
),

)
SELECT *
FROM order_metrics
ORDER BY merchant_id;
'''

# 执行 SQL
order_feature_df = sqldf(sql_order_features, globals())
print('订单特征示例：')
display(order_feature_df.head())


订单特征示例：


,merchant_id,orders_last_7d,days_since_last_order,order_trend_ratio,gmv_28d,orders_28d,aov_28d,unique_customers_28d,new_customer_ratio_28d,cancellation_rate_28d,merchant_delay_rate_28d,avg_prep_time_28d,late_delivery_rate_28d,refund_rate_28d,avg_rating_28d,one_star_rate_28d,complaint_rate_28d,repeat_customer_rate_28d,commission_paid_28d,profit_margin_28d,promo_cost_ratio_28d,dispute_rate_28d,payment_failed_rate_28d
0,M00001,3,2,1.090909,267.91,10,26.791000,10,0.500000,0.000000,0.000,19.845000,0.200000,0.000000,4.200000,0.0,0.000000,0.500000,61.72,0.205853,0.068269,0.000000,0.0
1,M00002,1,6,0.500000,242.54,7,34.648571,7,0.142857,0.166667,0.000,21.423333,0.285714,0.000000,4.500000,0.0,0.000000,0.714286,54.73,0.065639,0.133916,0.000000,0.0
2,M00003,0,33,0.000000,0.00,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,NaN,NaN,NaN,NaN
3,M00004,0,7,0.000000,373.24,17,21.955294,17,0.294118,0.000000,0.000,21.838235,0.352941,0.117647,4.235294,0.0,0.117647,0.705882,89.85,0.222484,0.005734,0.058824,0.0
4,M00005,1,2,0.571429,233.97,8,29.246250,8,0.125000,0.000000,0.125,17.576250,0.125000,0.000000,4.750000,0.0,0.000000,0.875000,54.72,0.125358,0.110741,0.125000,0.0


In [ ]:
from pandasql import sqldf
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

merchant_dim_df = pd.read_csv(data_dir / 'merchant_dim.csv')
merchant_daily_df = pd.read_csv(data_dir / 'merchant_daily_sample.csv')
orders_fact_df = pd.read_csv(data_dir / 'orders_sample.csv')
support_fact_df = pd.read_csv(data_dir / 'support_fact.csv')
promotion_fact_df = pd.read_csv(data_dir / 'promotion_fact.csv')

observation_date = '2026-06-30'

sql_all_features = '''
WITH params AS (
    SELECT DATE('2026-06-30') AS obs_date
),

order_metrics AS (
    SELECT
        o.merchant_id,
        SUM(CASE WHEN o.order_date > DATE(p.obs_date, '-7 days') AND o.order_date <= p.obs_date THEN 1 ELSE 0 END) AS orders_last_7d,
        CAST(julianday(p.obs_date) - julianday(MAX(o.order_date)) AS INTEGER) AS days_since_last_order,
        SUM(CASE WHEN o.order_date > DATE(p.obs_date, '-7 days') AND o.order_date <= p.obs_date THEN 1 ELSE 0 END) * 1.0
            / NULLIF(
                SUM(CASE WHEN o.order_date > DATE(p.obs_date, '-35 days') AND o.order_date <= DATE(p.obs_date, '-7 days') THEN 1 ELSE 0 END) / 4.0
              , 0) AS order_trend_ratio,
        SUM(CASE WHEN o.order_date > DATE(p.obs_date, '-28 days') AND o.order_date <= p.obs_date THEN o.gross_sales ELSE 0 END) AS gmv_28d,
        SUM(CASE WHEN o.order_date > DATE(p.obs_date, '-28 days') AND o.order_date <= p.obs_date THEN 1 ELSE 0 END) AS orders_28d,
        SUM(CASE WHEN o.order_date > DATE(p.obs_date, '-28 days') AND o.order_date <= p.obs_date THEN o.gross_sales ELSE 0 END) * 1.0
            / NULLIF(SUM(CASE WHEN o.order_date > DATE(p.obs_date, '-28 days') AND o.order_date <= p.obs_date THEN 1 ELSE 0 END), 0) AS aov_28d,
        COUNT(DISTINCT CASE WHEN o.order_date > DATE(p.obs_date, '-28 days') AND o.order_date <= p.obs_date THEN o.customer_id END) AS unique_customers_28d,
        SUM(CASE WHEN o.order_date > DATE(p.obs_date, '-28 days') AND o.order_date <= p.obs_date THEN o.new_customer_flag ELSE 0 END) * 1.0
            / NULLIF(COUNT(DISTINCT CASE WHEN o.order_date > DATE(p.obs_date, '-28 days') AND o.order_date <= p.obs_date THEN o.customer_id END), 0) AS new_customer_ratio_28d,
        SUM(CASE WHEN o.order_date > DATE(p.obs_date, '-28 days') AND o.order_date <= p.obs_date THEN o.cancelled_flag ELSE 0 END) * 1.0
            / NULLIF(SUM(CASE WHEN o.order_date > DATE(p.obs_date, '-28 days') AND o.order_date <= p.obs_date THEN o.accepted_flag ELSE 0 END), 0) AS cancellation_rate_28d,
        SUM(CASE WHEN o.order_date > DATE(p.obs_date, '-28 days') AND o.order_date <= p.obs_date THEN o.merchant_delay_flag ELSE 0 END) * 1.0
            / NULLIF(SUM(CASE WHEN o.order_date > DATE(p.obs_date, '-28 days') AND o.order_date <= p.obs_date THEN 1 ELSE 0 END), 0) AS merchant_delay_rate_28d,
        AVG(CASE WHEN o.order_date > DATE(p.obs_date, '-28 days') AND o.order_date <= p.obs_date THEN o.prep_time_minutes END) AS avg_prep_time_28d,
        SUM(CASE WHEN o.order_date > DATE(p.obs_date, '-28 days') AND o.order_date <= p.obs_date THEN o.late_delivery_flag ELSE 0 END) * 1.0
            / NULLIF(SUM(CASE WHEN o.order_date > DATE(p.obs_date, '-28 days') AND o.order_date <= p.obs_date THEN 1 ELSE 0 END), 0) AS late_delivery_rate_28d,
        SUM(CASE WHEN o.order_date > DATE(p.obs_date, '-28 days') AND o.order_date <= p.obs_date THEN o.refund_requested_flag ELSE 0 END) * 1.0
            / NULLIF(SUM(CASE WHEN o.order_date > DATE(p.obs_date, '-28 days') AND o.order_date <= p.obs_date THEN 1 ELSE 0 END), 0) AS refund_rate_28d,
        AVG(CASE WHEN o.order_date > DATE(p.obs_date, '-28 days') AND o.order_date <= p.obs_date THEN o.customer_rating END) AS avg_rating_28d,
        SUM(CASE WHEN o.order_date > DATE(p.obs_date, '-28 days') AND o.order_date <= p.obs_date AND o.customer_rating = 1 THEN 1 ELSE 0 END) * 1.0
            / NULLIF(SUM(CASE WHEN o.order_date > DATE(p.obs_date, '-28 days') AND o.order_date <= p.obs_date AND o.customer_rating IS NOT NULL THEN 1 ELSE 0 END), 0) AS one_star_rate_28d,
        SUM(CASE WHEN o.order_date > DATE(p.obs_date, '-28 days') AND o.order_date <= p.obs_date THEN o.complaint_flag ELSE 0 END) * 1.0
            / NULLIF(SUM(CASE WHEN o.order_date > DATE(p.obs_date, '-28 days') AND o.order_date <= p.obs_date THEN 1 ELSE 0 END), 0) AS complaint_rate_28d,
        SUM(CASE WHEN o.order_date > DATE(p.obs_date, '-28 days') AND o.order_date <= p.obs_date THEN o.repeat_customer_flag ELSE 0 END) * 1.0
            / NULLIF(SUM(CASE WHEN o.order_date > DATE(p.obs_date, '-28 days') AND o.order_date <= p.obs_date THEN 1 ELSE 0 END), 0) AS repeat_customer_rate_28d,
        SUM(CASE WHEN o.order_date > DATE(p.obs_date, '-28 days') AND o.order_date <= p.obs_date THEN o.commission_amount ELSE 0 END) AS commission_paid_28d,
        SUM(CASE WHEN o.order_date > DATE(p.obs_date, '-28 days') AND o.order_date <= p.obs_date THEN o.merchant_profit ELSE 0 END) * 1.0
            / NULLIF(SUM(CASE WHEN o.order_date > DATE(p.obs_date, '-28 days') AND o.order_date <= p.obs_date THEN o.gross_sales ELSE 0 END), 0) AS profit_margin_28d,
        SUM(CASE WHEN o.order_date > DATE(p.obs_date, '-28 days') AND o.order_date <= p.obs_date THEN o.merchant_dispute_flag ELSE 0 END) * 1.0
            / NULLIF(SUM(CASE WHEN o.order_date > DATE(p.obs_date, '-28 days') AND o.order_date <= p.obs_date THEN 1 ELSE 0 END), 0) AS dispute_rate_28d,
        SUM(CASE WHEN o.order_date > DATE(p.obs_date, '-28 days') AND o.order_date <= p.obs_date THEN o.payment_failed_flag ELSE 0 END) * 1.0
            / NULLIF(SUM(CASE WHEN o.order_date > DATE(p.obs_date, '-28 days') AND o.order_date <= p.obs_date THEN 1 ELSE 0 END), 0) AS payment_failed_rate_28d
    FROM orders_fact_df o, params p
    GROUP BY o.merchant_id
),

merchant_info AS (
    SELECT
        m.merchant_id,
        CAST(julianday(p.obs_date) - julianday(m.join_date) AS INTEGER) AS account_age_days,
        m.segment,
        m.city,
        m.menu_ready AS is_menu_ready,
        m.competitor_presence
    FROM merchant_dim_df m, params p
),

merchant_activity AS (
    SELECT
        d.merchant_id,
        SUM(CASE WHEN d.date > DATE(p.obs_date, '-30 days') AND d.date <= p.obs_date THEN d.is_open ELSE 0 END) AS store_open_days_30d,
        SUM(CASE WHEN d.date > DATE(p.obs_date, '-30 days') AND d.date <= p.obs_date THEN d.actual_open_hours ELSE 0 END) * 1.0
            / NULLIF(SUM(CASE WHEN d.date > DATE(p.obs_date, '-30 days') AND d.date <= p.obs_date THEN d.scheduled_hours ELSE 0 END), 0) AS business_hours_consistency_30d,
        SUM(CASE WHEN d.date > DATE(p.obs_date, '-30 days') AND d.date <= p.obs_date THEN d.temporary_closure_flag ELSE 0 END) AS temporary_closure_days_30d
    FROM merchant_daily_df d, params p
    GROUP BY d.merchant_id
),

support_metrics AS (
    SELECT
        s.merchant_id,
        SUM(CASE WHEN s.created_at > DATE(p.obs_date, '-30 days') AND s.created_at <= p.obs_date THEN 1 ELSE 0 END) AS support_ticket_count_30d,
        AVG(CASE WHEN s.created_at > DATE(p.obs_date, '-30 days') AND s.created_at <= p.obs_date THEN s.resolution_hours END) AS avg_resolution_hours_30d
    FROM support_fact_df s, params p
    GROUP BY s.merchant_id
),

promotion_metrics AS (
    SELECT
        pr.merchant_id,
        SUM(CASE WHEN pr.start_date > DATE(p.obs_date, '-30 days') AND pr.start_date <= p.obs_date THEN 1 ELSE 0 END) AS promo_participation_count_30d,
        SUM(CASE WHEN pr.start_date > DATE(p.obs_date, '-30 days') AND pr.start_date <= p.obs_date THEN pr.promo_spend ELSE 0 END) AS promo_spend_30d,
        SUM(CASE WHEN pr.start_date > DATE(p.obs_date, '-30 days') AND pr.start_date <= p.obs_date THEN pr.promotion_return ELSE 0 END) * 1.0
            / NULLIF(SUM(CASE WHEN pr.start_date > DATE(p.obs_date, '-30 days') AND pr.start_date <= p.obs_date THEN pr.promo_spend ELSE 0 END), 0) AS promo_roi_30d
    FROM promotion_fact_df pr, params p
    GROUP BY pr.merchant_id
)

SELECT
    mi.merchant_id,

    -- 商家信息
    mi.account_age_days,
    mi.segment,
    mi.city,

    -- 订单表现 Order Performance
    om.orders_last_7d,
    om.days_since_last_order,
    om.order_trend_ratio,
    om.gmv_28d,
    om.aov_28d,
    om.unique_customers_28d,
    om.new_customer_ratio_28d,

    -- 商家质量与经营活跃度
    mi.is_menu_ready,
    pm.promo_participation_count_30d,
    ma.store_open_days_30d,
    ma.business_hours_consistency_30d,
    ma.temporary_closure_days_30d,

    -- 运营质量 Operational Quality
    om.cancellation_rate_28d,
    om.merchant_delay_rate_28d,
    om.avg_prep_time_28d,
    om.late_delivery_rate_28d,
    om.refund_rate_28d,

    -- 客户体验 Customer Experience
    om.avg_rating_28d,
    om.one_star_rate_28d,
    om.complaint_rate_28d,
    om.repeat_customer_rate_28d,

    -- 财务表现 Financial Health
    om.commission_paid_28d,
    om.profit_margin_28d,
    pm.promo_spend_30d * 1.0 / NULLIF(om.gmv_28d, 0) AS promo_cost_ratio,
    pm.promo_roi_30d,
    om.dispute_rate_28d,

    -- 平台关系 Platform Relationship
    sm.support_ticket_count_30d,
    sm.avg_resolution_hours_30d,
    om.payment_failed_rate_28d,
    mi.competitor_presence

FROM merchant_info mi
LEFT JOIN order_metrics om ON mi.merchant_id = om.merchant_id
LEFT JOIN merchant_activity ma ON mi.merchant_id = ma.merchant_id
LEFT JOIN support_metrics sm ON mi.merchant_id = sm.merchant_id
LEFT JOIN promotion_metrics pm ON mi.merchant_id = pm.merchant_id
ORDER BY mi.merchant_id;
'''

feature_df = sqldf(sql_all_features, globals())

count_cols_fillna_zero = [
    'promo_participation_count_30d',
    'support_ticket_count_30d',
]
feature_df[count_cols_fillna_zero] = feature_df[count_cols_fillna_zero].fillna(0)

print('特征表规模:', feature_df.shape)
display(feature_df.head())

特征表规模: (2000, 34)


,merchant_id,account_age_days,segment,city,orders_last_7d,days_since_last_order,order_trend_ratio,gmv_28d,aov_28d,unique_customers_28d,new_customer_ratio_28d,is_menu_ready,promo_participation_count_30d,store_open_days_30d,business_hours_consistency_30d,temporary_closure_days_30d,cancellation_rate_28d,merchant_delay_rate_28d,avg_prep_time_28d,late_delivery_rate_28d,refund_rate_28d,avg_rating_28d,one_star_rate_28d,complaint_rate_28d,repeat_customer_rate_28d,commission_paid_28d,profit_margin_28d,promo_cost_ratio,promo_roi_30d,dispute_rate_28d,support_ticket_count_30d,avg_resolution_hours_30d,payment_failed_rate_28d,competitor_presence
0,M00001,680,SMB,Oakland,3.0,2.0,1.090909,267.91,26.791000,10.0,0.500000,1,0.0,0,None,0,0.000000,0.000,19.845000,0.200000,0.000000,4.200000,0.0,0.000000,0.500000,61.72,0.205853,0.0,None,0.000000,0.0,NaN,0.0,0
1,M00002,720,SMB,San Jose,1.0,6.0,0.500000,242.54,34.648571,7.0,0.142857,1,0.0,0,None,0,0.166667,0.000,21.423333,0.285714,0.000000,4.500000,0.0,0.000000,0.714286,54.73,0.065639,0.0,None,0.000000,0.0,NaN,0.0,0
2,M00003,917,SMB,San Francisco,0.0,33.0,0.000000,0.00,NaN,0.0,NaN,1,0.0,0,None,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,NaN,NaN,None,NaN,0.0,NaN,NaN,1
3,M00004,1943,Mid-Market,Oakland,0.0,7.0,0.000000,373.24,21.955294,17.0,0.294118,1,0.0,0,None,0,0.000000,0.000,21.838235,0.352941,0.117647,4.235294,0.0,0.117647,0.705882,89.85,0.222484,0.0,None,0.058824,1.0,NaN,0.0,1
4,M00005,1210,SMB,San Francisco,1.0,2.0,0.571429,233.97,29.246250,8.0,0.125000,1,0.0,0,None,0,0.000000,0.125,17.576250,0.125000,0.000000,4.750000,0.0,0.000000,0.875000,54.72,0.125358,0.0,None,0.125000,0.0,NaN,0.0,0


In [40]:
orders28 = '''
WITH params AS (
    SELECT DATE('2026-06-30') AS obs_date
)
SELECT SUM(CASE WHEN o.order_date > DATE(p.obs_date, '-28 days') AND o.order_date <= p.obs_date THEN 1 ELSE 0 END) AS orders_28d
FROM orders_fact_df o, params p
GROUP BY o.merchant_id
'''
f = sqldf(orders28, globals())
display(f[f['orders_28d'].isna()])

,orders_28d
